**Linda Zier**

**ST 554 ~ Final Project**

**04/30/2026**

# Goal

The goal of this project was to use Apache Spark to fit a machine learning model and apply it to streaming data. All project files were housed in a GitHub repository, including a Jupyter notebook and a Python producer script. In the notebook, I fit an elastic net regression model using PySpark's MLlib module to predict power consumption in Zone 3 of Tetouan City from weather and time variables. I then simulated a data stream by writing a separate Python script that periodically wrote batches of data to a monitored folder. As data arrived in the stream, I used the fitted model to generate predictions and wrote the results out to the console.

### Data

The data used in this project is modified from the UCI Machine Learning Repository and is available at https://www4.stat.ncsu.edu/~online/datasets/power_ml_data.csv. The dataset contains measurements from Tetouan City relating power consumption across three zones to factors such as temperature, humidity, wind speed, diffuse solar flows, and time of day. I used this dataset to fit my model, treating Power Zone 3 consumption as the response variable. A separate streaming dataset (power_streaming_data.csv) was used to simulate incoming data. My producer script repeatedly sampled from this file and wrote batches to a monitored folder where the fitted model generated predictions on the arriving data.

# Fitting the Model

I created a Jupyter notebook for the model fitting part and the streaming part below. I read the data into a standard pandas data frame using the pd.read_csv() function and converted this to a spark data frame.

In [1]:
import pandas as pd
from pyspark.sql import SparkSession
from pyspark.ml import Pipeline
from pyspark.ml.feature import SQLTransformer, VectorAssembler, Binarizer, \
                               OneHotEncoder, PCA
from pyspark.ml.regression import LinearRegression
from pyspark.ml.tuning import CrossValidator, ParamGridBuilder
from pyspark.sql.functions import col
from pyspark.ml.evaluation import RegressionEvaluator

spark = SparkSession.builder.getOrCreate()

# read in data/power_ml_data.csv as pandas dataframe
powerDF=pd.read_csv("data/power_ml_data.csv")

#convert to spark dataframe
powerDF=spark.createDataFrame(powerDF)


Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/04/30 10:39:36 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable
26/04/30 10:39:36 WARN Utils: Service 'SparkUI' could not bind on port 4040. Attempting port 4041.
26/04/30 10:39:36 WARN Utils: Service 'SparkUI' could not bind on port 4041. Attempting port 4042.


### Creating the Pipeline

I fit an elastic net model using cross validation with the steps below. The transformations used an MLlib function that was put into a pipeline.

*   I used an SQL transformer to cast the hour variable as DoubleType.
*   I binarized the Hour column based on the column being less than 6.5 or not (night vs day essentially).
*   The month column was one-hot encoded.
*   I ran a Principle Component Analysis (PCA) on the Temperature, Humidity, Wind_Speed, General_Diffuse_Flows, and Diffuse_Flows columns. I did this by:

        - First using a VectorAssembler() to place these variables in a single column for use with the PCA() estimator  
        
        - Then applying a PCA transformer for use in our pipeline using two principle components.
        
        
*   I renamed our response variable Power_Zone_3 as label.

*   I used VectorAssembler() to put my predictors into a features vector. The predictors were:

    – Two fitted PCA features
    
    – Binary Hour variable
    
    – Power_Zone_1
    
    – Power_Zone_2
    
    – Month indicator variables
    
I then built my pipeline for the transformations.
    


In [2]:
# check the data types
powerDF.printSchema()

# cast the hour as double since it is a long
sql = SQLTransformer(statement = '''
                     SELECT *, 
                     CAST(Hour AS DOUBLE) AS HourD FROM __THIS__
                     ''')

# binarize night vs day
binarizer = Binarizer(threshold=6.5, inputCol="HourD", outputCol="Hour_bin")

# one hot encode month
ohe = OneHotEncoder(inputCols=["Month"], outputCols=["Month_ohe"])

#VectorAssembler to bundle features together for pca
pca_assembler = VectorAssembler(
    inputCols=["Temperature", "Humidity", "Wind_Speed", 
               "General_Diffuse_Flows", "Diffuse_Flows"],
    outputCol="pca_input")

# pca with 2 components
pca = PCA(k=2, inputCol="pca_input", outputCol="pca_features")

# response variable to label
sql_label= SQLTransformer(statement = '''
                          SELECT *,
                          Power_Zone_3 AS label FROM __THIS__
                          ''')
# assemble final features
assembler = VectorAssembler(
    inputCols=["pca_features", "Hour_bin", "Power_Zone_1", 
               "Power_Zone_2","Month_ohe"],
    outputCol="features")

print("TRANSFORMATIONS COMPLETE")   
                     

root
 |-- Temperature: double (nullable = true)
 |-- Humidity: double (nullable = true)
 |-- Wind_Speed: double (nullable = true)
 |-- General_Diffuse_Flows: double (nullable = true)
 |-- Diffuse_Flows: double (nullable = true)
 |-- Power_Zone_1: double (nullable = true)
 |-- Power_Zone_2: double (nullable = true)
 |-- Power_Zone_3: double (nullable = true)
 |-- Month: long (nullable = true)
 |-- Hour: long (nullable = true)

TRANSFORMATIONS COMPLETE


In [3]:
from pyspark.ml import Pipeline

#build pipeline
pipeline= Pipeline(stages = [sql, binarizer, ohe, pca_assembler, 
                             pca, sql_label, assembler])
fittedPipeline = pipeline.fit(powerDF)
transformedDF=fittedPipeline.transform(powerDF)

print("PIPELINE COMPLETE")

PIPELINE COMPLETE


### Fitting an Elastic Net Model

Next I used the CrossValidator and LinearRegression functions to fit an elastic net model. I searched over multiple combinations of regularization parameters, which control the strength of the penalty, and elastic net parameters, which control the mix between Lasso and Ridge. The model was fit using 5-fold cross validation with root mean square error (RMSE) as the criteria: I trained 5 separate models (one per fold) and averaged their RMSEs to evaluate each parameter combination. I then reported the optimal tuning parameter values and the CV error, which is the RMSE from the best model.


In [4]:

# setting up elastic net model
lr= LinearRegression(elasticNetParam=0.5)


#  grid for the regParam and elasticNetParam
paramGrid= ParamGridBuilder() \
    .addGrid(lr.regParam,[0, 0.05, 0.1, 0.25, 0.5, 0.75, 0.9, 0.95, 0.98, 0.99, 1]) \
    .addGrid(lr.elasticNetParam,[ 0, 0.05, 0.1, 0.25, 0.5, 0.75, 0.9, 0.95, 0.98, 0.99, 1]) \
    .build()

# 5-fold CV with rmse evaluator
cv=CrossValidator(estimator=lr,
                   estimatorParamMaps = paramGrid,  
                   evaluator = RegressionEvaluator(metricName= 'rmse'),
                   numFolds=5)

# fit the model
cvModel=cv.fit(transformedDF)

# Report the optimal values chosen for the tuning parameters
print("Optimal regularization parameter: ", cvModel.bestModel.getRegParam())
print("Optimal elastic net parameter: ", cvModel.bestModel.getElasticNetParam())

# report RMSE errors - if you want to see all 11 x 11 = 121 of them
# print("RMSE errors= ", cvModel.avgMetrics)

# report lowest RMSE
print("CV RMSE: ", min(cvModel.avgMetrics))


26/04/30 10:39:48 WARN SparkStringUtils: Truncated the string representation of a plan since it was too large. This behavior can be adjusted by setting 'spark.sql.debug.maxToStringFields'.
26/04/30 10:39:49 WARN Instrumentation: [d78ef27e] regParam is zero, which might cause numerical instability and overfitting.
26/04/30 10:39:50 WARN Instrumentation: [d78ef27e] Cholesky solver failed due to singular covariance matrix. Retrying with Quasi-Newton solver.
26/04/30 10:39:51 WARN Instrumentation: [2e0c37ef] regParam is zero, which might cause numerical instability and overfitting.
26/04/30 10:39:52 WARN Instrumentation: [2e0c37ef] Cholesky solver failed due to singular covariance matrix. Retrying with Quasi-Newton solver.
26/04/30 10:39:52 WARN Instrumentation: [356724ee] regParam is zero, which might cause numerical instability and overfitting.
26/04/30 10:39:52 WARN Instrumentation: [356724ee] Cholesky solver failed due to singular covariance matrix. Retrying with Quasi-Newton solver.
2

Optimal regularization parameter:  0.1
Optimal elastic net parameter:  0.25
CV RMSE:  2147.724094853242


### Training Set Evaluation
I reported the training set RMSE using our fitted model as a transformer and
evaluating on the entire training set. I then created a residual column (label - prediction) and printed the data frame with these residuals. I also printed a summary table as a sanity check to ensure the mean was near zero and the standard deviation was reasonable.

In [5]:
# report training set RMSE by using fitted model as a transformer
predictions = cvModel.transform(transformedDF)
trainRMSE = RegressionEvaluator(metricName= 'rmse').evaluate(predictions)
print("Training RMSE=", trainRMSE)

# create residual column and display results
print("Training set residuals and summary:")
predictions = predictions.withColumn("residual", col("label") - col("prediction"))
predictions.select("label", "prediction", "residual").show()

#sanity check on residual distribution
predictions.select("label", "prediction", "residual") \
    .summary("mean", "stddev", "min", "max") \
    .show()

Training RMSE= 2147.097381290536
Training set residuals and summary:
+-----------+------------------+------------------+
|      label|        prediction|          residual|
+-----------+------------------+------------------+
|20240.96386| 20880.33970130881|-639.3758413088108|
|20131.08434|18659.921478081902|1471.1628619180992|
|19668.43373|18204.427090265977|1464.0066397340233|
|18899.27711|17590.347680927865|1308.9294290721336|
|18442.40964|16996.978968117968|1445.4306718820335|
|18130.12048|16517.366359426618|1612.7541205733833|
|17945.06024|16092.934692961617|1852.1255470383821|
|17459.27711|15722.379081012821| 1736.898028987178|
|17025.54217|15270.731694846627|1754.8104751533738|
|16794.21687|14938.028946615765|1856.1879233842355|
|16638.07229|14652.143575261782|1985.9287147382183|
|16395.18072|14414.660768824788|1980.5199511752126|
|16117.59036|14082.559411468532|2035.0309485314683|
| 15822.6506|13624.562955558858|2198.0876444411424|
|15672.28916| 13450.06385226433| 2222.225307735

# Handling Streaming Data
I downloaded the streaming source file from https://www4.stat.ncsu.edu/~online/datasets/power_streaming_data.csv
and stored it in my final_project/data directory.  This file served as my source for random sampling in my producer script.
### Reading a Stream
I read in a stream in the form of .csv files. I created a folder (using mkdir in terminal) called streaming_data to serve as the monitored directory. The schema was set to that of the original data since that is what my incoming data would have the same structure and a header was assumed to be present.

In [6]:
# set the schema to that of the original data
stream_schema=powerDF.schema
print(stream_schema)

# set up readstream with a header
streamDF = spark.readStream.schema(stream_schema).option("header", True)\
           .csv("streaming_data")

#showing current working directory
#import os
#os.getcwd()

StructType([StructField('Temperature', DoubleType(), True), StructField('Humidity', DoubleType(), True), StructField('Wind_Speed', DoubleType(), True), StructField('General_Diffuse_Flows', DoubleType(), True), StructField('Diffuse_Flows', DoubleType(), True), StructField('Power_Zone_1', DoubleType(), True), StructField('Power_Zone_2', DoubleType(), True), StructField('Power_Zone_3', DoubleType(), True), StructField('Month', LongType(), True), StructField('Hour', LongType(), True)])



### Transform/Aggregation Step

In this code block I used the fitted model as a transformer to obtain predictions from the incoming data stream. First I created a residual column (label - prediction) as done in the previous section, returning only the label, prediction, and residual columns. Then, using a second transformation on the original stream, I renamed the response variable Power_Zone_3 to label, keeping all other columns intact. Finally I joined the two transformations together on the label variable using an inner join.

In [7]:
# ---Transformation 1:---

# apply transformer to the data stream
streamPredictions = cvModel.transform(fittedPipeline.transform(streamDF))

#add residual column = label - predictions
streamResiduals=streamPredictions.withColumn("residual",col("label")- col("prediction")) \
                                  .select("label","prediction", "residual")
                                              
# ---Transformation 2:---

# rename response to
streamLabeled=sql_label.transform(streamDF)

# --- Inner Join ---
streamJoined = streamResiduals.join(streamLabeled, on="label", how="inner")



### Writing Step
I wrote the joined stream to the console using append output mode, which outputs only newly arriving rows with each batch. I started the query and allowed it to run for 20 seconds before terminating it with query.stop().

In [37]:
# Write stream to console in append mode and start query
#query = streamJoined.writeStream.outputMode("append").format("console")

query = streamJoined.writeStream \
                    .outputMode("append") \
                    .format("console") \
                    .start()


26/04/30 12:30:01 WARN ResolveWriteToStream: Temporary checkpoint location created which is deleted normally when the query didn't fail: /tmp/temporary-0e5f3f20-86de-4f3c-85f0-f2776298930b. If it's required to delete it under any circumstances, please set spark.sql.streaming.forceDeleteTempCheckpointLocation to true. Important to know deleting temp checkpoint folder is best effort.
26/04/30 12:30:01 WARN ResolveWriteToStream: spark.sql.adaptive.enabled is not supported in streaming DataFrames/Datasets and will be disabled.


-------------------------------------------
Batch: 0
-------------------------------------------
+-----------+------------------+------------------+-----------+--------+----------+---------------------+-------------+------------+------------+------------+-----+----+
|      label|        prediction|          residual|Temperature|Humidity|Wind_Speed|General_Diffuse_Flows|Diffuse_Flows|Power_Zone_1|Power_Zone_2|Power_Zone_3|Month|Hour|
+-----------+------------------+------------------+-----------+--------+----------+---------------------+-------------+------------+------------+------------+-----+----+
|18696.68016|18593.409449611565|103.27071038843496|      23.91|   58.81|     4.925|                534.8|        59.71| 35239.86885| 21477.39938| 18696.68016|    5|  17|
|25347.46988| 24456.42903622597| 891.0408437740298|      12.61|    77.3|     0.088|                0.026|        0.148| 41061.26582| 26283.28267| 25347.46988|    1|  21|
|13968.72362|13036.785036855372| 931.9385831446289|  

-------------------------------------------
Batch: 1
-------------------------------------------
+-----------+------------------+------------------+-----------+--------+----------+---------------------+-------------+------------+------------+------------+-----+----+
|      label|        prediction|          residual|Temperature|Humidity|Wind_Speed|General_Diffuse_Flows|Diffuse_Flows|Power_Zone_1|Power_Zone_2|Power_Zone_3|Month|Hour|
+-----------+------------------+------------------+-----------+--------+----------+---------------------+-------------+------------+------------+------------+-----+----+
|37719.87461|31844.641125951835|5875.2334840481635|      25.83|    82.9|     4.926|                 15.8|        12.59| 46923.86238| 31232.94615| 37719.87461|    8|  19|
| 11572.5228|14370.626030056113|-2798.103230056113|      22.63|    74.1|     0.084|                210.0|        138.0| 35996.84902| 22899.58506|  11572.5228|   10|  10|
|14397.05822| 16743.10354309166|-2346.045323091659|  

-------------------------------------------
Batch: 2
-------------------------------------------
+-----------+------------------+-------------------+-----------+--------+----------+---------------------+-------------+------------+------------+------------+-----+----+
|      label|        prediction|           residual|Temperature|Humidity|Wind_Speed|General_Diffuse_Flows|Diffuse_Flows|Power_Zone_1|Power_Zone_2|Power_Zone_3|Month|Hour|
+-----------+------------------+-------------------+-----------+--------+----------+---------------------+-------------+------------+------------+------------+-----+----+
|20192.35955|13843.418533286569|  6348.941016713432|      22.77|   51.82|     4.925|                135.5|        138.3| 33247.43363| 16817.46362| 20192.35955|    9|  18|
|41302.09205|  33539.8902687031|  7762.201781296899|       26.9|    79.0|     4.916|                104.1|         92.2| 44242.92359| 27865.82278| 41302.09205|    7|  19|
|17106.50602| 18608.57932707874|-1502.0733070787

-------------------------------------------
Batch: 3
-------------------------------------------
+-----------+------------------+-------------------+-----------+--------+----------+---------------------+-------------+------------+------------+------------+-----+----+
|      label|        prediction|           residual|Temperature|Humidity|Wind_Speed|General_Diffuse_Flows|Diffuse_Flows|Power_Zone_1|Power_Zone_2|Power_Zone_3|Month|Hour|
+-----------+------------------+-------------------+-----------+--------+----------+---------------------+-------------+------------+------------+------------+-----+----+
|36877.24138|  33758.1917473104| 3119.0496326896027|      27.64|   49.16|     4.904|                 36.4|        31.63| 49736.73696| 34129.67265| 36877.24138|    8|  19|
|15173.25228|13114.489113155361| 2058.7631668446393|      21.82|   60.33|     0.098|                0.048|        0.174| 32858.46827| 20968.87967| 15173.25228|   10|  23|
| 24340.0627|20464.345286705808|  3875.717413294

...

-------------------------------------------
Batch: 5
-------------------------------------------
+-----------+------------------+------------------+-----------+--------+----------+---------------------+-------------+------------+------------+------------+-----+----+
|      label|        prediction|          residual|Temperature|Humidity|Wind_Speed|General_Diffuse_Flows|Diffuse_Flows|Power_Zone_1|Power_Zone_2|Power_Zone_3|Month|Hour|
+-----------+------------------+------------------+-----------+--------+----------+---------------------+-------------+------------+------------+------------+-----+----+
|27969.40439|  25823.1300263987|   2146.2743636013|      27.31|    71.6|     4.903|                227.0|        177.0| 39066.99223| 28165.15312| 27969.40439|    8|  17|
|20892.50255| 21851.06047483618|-958.5579248361792|      21.61|   58.89|     0.267|                0.055|        0.156| 43690.61947| 26034.51143| 20892.50255|    9|  22|
|12722.18845|13524.901396665447|-802.7129466654478|  

-------------------------------------------
Batch: 6
-------------------------------------------
+-----------+------------------+-------------------+-----------+--------+----------+---------------------+-------------+------------+------------+------------+-----+----+
|      label|        prediction|           residual|Temperature|Humidity|Wind_Speed|General_Diffuse_Flows|Diffuse_Flows|Power_Zone_1|Power_Zone_2|Power_Zone_3|Month|Hour|
+-----------+------------------+-------------------+-----------+--------+----------+---------------------+-------------+------------+------------+------------+-----+----+
|19626.01824|  20956.7496787699| -1330.731438769897|      20.25|   69.13|     0.074|                0.084|          0.1| 44964.55142| 25700.41494| 19626.01824|   10|  20|
|10019.69356|11071.952228542052| -1052.258668542052|      20.35|   68.32|     0.293|                0.051|        0.152| 24983.36283| 14807.90021| 10019.69356|    9|   5|
|17661.68675| 13382.94916613188|   4278.73758386

-------------------------------------------
Batch: 7
-------------------------------------------
+-----------+------------------+-------------------+-----------+--------+----------+---------------------+-------------+------------+------------+------------+-----+----+
|      label|        prediction|           residual|Temperature|Humidity|Wind_Speed|General_Diffuse_Flows|Diffuse_Flows|Power_Zone_1|Power_Zone_2|Power_Zone_3|Month|Hour|
+-----------+------------------+-------------------+-----------+--------+----------+---------------------+-------------+------------+------------+------------+-----+----+
|28567.27273|26354.509039634046|  2212.763690365955|      17.11|   68.95|     0.085|                0.026|        0.156| 43141.09795| 21731.97556| 28567.27273|    4|  20|
|27293.09091|25509.019373462987| 1784.0715365370124|      15.89|    84.6|     0.072|                0.033|        0.178| 41572.44349| 22699.79633| 27293.09091|    4|  21|
|25248.90282| 24092.02521785478| 1156.8776021452

-------------------------------------------
Batch: 8
-------------------------------------------
+-----------+------------------+-------------------+-----------+--------+----------+---------------------+-------------+------------+------------+------------+-----+----+
|      label|        prediction|           residual|Temperature|Humidity|Wind_Speed|General_Diffuse_Flows|Diffuse_Flows|Power_Zone_1|Power_Zone_2|Power_Zone_3|Month|Hour|
+-----------+------------------+-------------------+-----------+--------+----------+---------------------+-------------+------------+------------+------------+-----+----+
|10953.25301| 9358.175225616325|  1595.077784383675|      15.25|    88.9|      0.07|                0.029|        0.167| 21526.15385| 16910.33058| 10953.25301|   11|   3|
|14498.31325|  14672.0814484836|-173.76819848359992|      14.19|   64.44|     0.081|                104.6|         93.5| 27973.67089| 13619.45289| 14498.31325|    1|  10|
|19093.55649| 22493.98786594184|  -3400.43137594

-------------------------------------------
Batch: 9
-------------------------------------------
+-----------+------------------+-------------------+-----------+--------+----------+---------------------+-------------+------------+------------+------------+-----+----+
|      label|        prediction|           residual|Temperature|Humidity|Wind_Speed|General_Diffuse_Flows|Diffuse_Flows|Power_Zone_1|Power_Zone_2|Power_Zone_3|Month|Hour|
+-----------+------------------+-------------------+-----------+--------+----------+---------------------+-------------+------------+------------+------------+-----+----+
|10457.87234|12981.152485930665|-2523.2801459306647|      19.32|    92.0|     4.922|                221.4|        130.0| 34036.93654| 21024.89627| 10457.87234|   10|   9|
|9450.180072| 7919.628075079121|  1530.551996920878|      16.43|   64.55|     0.081|                0.033|        0.141|  22326.9962| 17681.49739| 9450.180072|   12|   5|
|10160.96386|12350.091969772953|-2189.1281097729

In [38]:
# stop querying
query.stop()

### the other commands below were used when repeatedly testing
#for s in spark.streams.active:
#    s.stop()
    
#for f in os.listdir("streaming_data"):
#    os.remove(f"streaming_data/{f}")

26/04/30 12:31:57 ERROR WriteToDataSourceV2Exec: Data source write support MicroBatchWrite[epoch: 10, writer: ConsoleWriter[numRows=20, truncate=true]] is aborting.
26/04/30 12:31:57 ERROR WriteToDataSourceV2Exec: Data source write support MicroBatchWrite[epoch: 10, writer: ConsoleWriter[numRows=20, truncate=true]] aborted.
26/04/30 12:31:57 WARN Shell: Interrupted while joining on: Thread[Thread-488018,5,main]
java.lang.InterruptedException
	at java.base/java.lang.Object.wait(Native Method)
	at java.base/java.lang.Thread.join(Thread.java:1313)
	at java.base/java.lang.Thread.join(Thread.java:1381)
	at org.apache.hadoop.util.Shell.joinThread(Shell.java:1103)
	at org.apache.hadoop.util.Shell.runCommand(Shell.java:1063)
	at org.apache.hadoop.util.Shell.run(Shell.java:959)
	at org.apache.hadoop.util.Shell$ShellCommandExecutor.execute(Shell.java:1282)
	at org.apache.hadoop.util.Shell.execCommand(Shell.java:1377)
	at org.apache.hadoop.util.Shell.execCommand(Shell.java:1359)
	at org.apache.ha